# Plot Benchmark Results

Notebook nay chi doc CSV benchmark da co san va ve chart, khong chay infer lai.


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False


In [ ]:
# ===== User Config =====
BENCH_ROOT = Path(r"D:\Homework\BackEnd\ModelDetectApi\benchmark_outputs")
RUN_DIR: Path | None = None  # Dat thu muc cu the neu muon; None -> lay run moi nhat

if RUN_DIR is None:
    run_dirs = sorted([p for p in BENCH_ROOT.iterdir() if p.is_dir()])
    if not run_dirs:
        raise RuntimeError(f"Khong tim thay run nao trong {BENCH_ROOT}")
    RUN_DIR = run_dirs[-1]

SUMMARY_CSV = RUN_DIR / "benchmark_summary.csv"
COMPARE_CSV = RUN_DIR / "benchmark_compare_trt_vs_torch.csv"

if not SUMMARY_CSV.exists():
    raise RuntimeError(f"Khong tim thay file: {SUMMARY_CSV}")

PLOT_DIR = RUN_DIR / "plots_only"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_DIR: {RUN_DIR}")
print(f"SUMMARY_CSV: {SUMMARY_CSV}")
print(f"PLOT_DIR: {PLOT_DIR}")


In [ ]:
df = pd.read_csv(SUMMARY_CSV)
df_ok = df[df["status"] == "ok"].copy()
if df_ok.empty:
    raise RuntimeError("Khong co run status=ok de ve chart")

df_ok["case"] = df_ok["model_script"] + "\n" + df_ok["ckpt_stem"]
df_ok = df_ok.sort_values(["model_script", "ckpt_stem", "backend"]).reset_index(drop=True)

display(df_ok)


In [ ]:
if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
items = [
    ("accuracy_top1", "Top-1 Accuracy"),
    ("infer_mean_ms", "Inference Mean (ms)"),
    ("infer_p95_ms", "Inference P95 (ms)"),
    ("startup_ms", "Startup Time (ms)"),
]

for ax, (metric, title) in zip(axes.flatten(), items):
    if HAS_SEABORN:
        sns.barplot(data=df_ok, x="case", y=metric, hue="backend", ax=ax)
    else:
        pivot = df_ok.pivot_table(index="case", columns="backend", values=metric, aggfunc="mean")
        pivot.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Model / Checkpoint")
    ax.tick_params(axis="x", labelrotation=30)

plt.tight_layout()
out = PLOT_DIR / "overview_2x2.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")


In [ ]:
# Detail chart: latency p95 (de doc hon)
fig, ax = plt.subplots(figsize=(18, 6))
if HAS_SEABORN:
    sns.barplot(data=df_ok, x="case", y="infer_p95_ms", hue="backend", ax=ax)
else:
    pivot = df_ok.pivot_table(index="case", columns="backend", values="infer_p95_ms", aggfunc="mean")
    pivot.plot(kind="bar", ax=ax)
ax.set_title("Inference P95 (ms) by checkpoint")
ax.set_xlabel("Model / Checkpoint")
ax.tick_params(axis="x", labelrotation=30)

plt.tight_layout()
out = PLOT_DIR / "latency_p95_detail.png"
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")


In [ ]:
# Compare table (neu file da ton tai)
if COMPARE_CSV.exists():
    df_cmp = pd.read_csv(COMPARE_CSV)
    display(df_cmp)
else:
    print(f"Khong tim thay {COMPARE_CSV}. Ban co the chay notebook benchmark de tao file nay.")
